In [2]:
import pandas as pd
import ast
import numpy as np
import os
from itertools import islice
from tqdm import tqdm
import json
from transformers import AutoTokenizer, AutoModel


/opt/anaconda3/envs/preclinical/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Entity normalization tests

### Load embeddings

In [5]:
directory_path = "./data/umls/embeddings"
batch_name_prefix = "UMLS_emb"

# List all files in the directory that match the pattern
files = [f for f in os.listdir(directory_path) if f.startswith(f'{batch_name_prefix}_batch_') and f.endswith('.npy')]
print(files)
# Sort files to maintain the order, especially important if the batch index is used in processing
files.sort()

# Initialize an empty list to hold the data from each file
all_data = []

# Load each file and append the data to the list
for file in files:
    file_path = os.path.join(directory_path, file)
    data = np.load(file_path)
    all_data.append(data)

all_reps_emb_full = np.concatenate(all_data, axis=0)

['UMLS_emb_batch_3.npy', 'UMLS_emb_batch_2.npy', 'UMLS_emb_batch_0.npy', 'UMLS_emb_batch_1.npy', 'UMLS_emb_batch_4.npy']


In [6]:
all_reps_emb_full.shape

(474316, 768)

In [7]:
with open("data/umls/umls_term_id_pairs.json", "r") as f:
    term_id_pairs = json.load(f)

In [8]:
len(term_id_pairs)

474316

# Test mappings

In [15]:
from neural_based_nen import map_query_to_terminology

In [16]:
tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")

In [17]:
len(all_reps_emb_full), len(term_id_pairs)

(474316, 474316)

In [18]:
# Example usage
dist_threshold = 15
embeddings_to_use = all_reps_emb_full
corresponding_term_id = term_id_pairs

In [31]:
query = "SDZ RAD"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C5200978
Predicted label: SDZ RAD
Distance: 0.0
Canonical form: SDZ RAD
Nearest 3:  [['SDZ RAD', 'C5200978'], ['SDR composite', 'C4505781'], ['SDZ RAD 223-756', 'C1740205'], ['RSD 1000', 'C0765569'], ['dodecyl sulfate, rubidium salt', 'C0954606']]


In [25]:
query = "Tadalafil"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C1176316
Predicted label: Tadalafil
Distance: 0.0
Canonical form: Tadalafil
Nearest 3:  [['Tadalafil', 'C1176316'], ['aminotadalafil', 'C2935567'], ['Tadliq', 'C5705462'], ['QuiXfil', 'C1721819'], ['Avanafil', 'C2698280']]


In [27]:
query = "Alyq"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C4759019
Predicted label: Alyq
Distance: 0.0
Canonical form: Alyq
Nearest 3:  [['Alyq', 'C4759019'], ['Alora', 'C0718414'], ['Alogabat', 'C5669632'], ['Alo', 'C3474139'], ['ALXN1101', 'C5545185']]


In [20]:
query = "interferon beta"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C0015980
Predicted label: beta Interferon
Distance: 3.9498
Canonical form: beta Interferon
Nearest 3:  [['beta Interferon', 'C0015980'], ['Interferon beta (recombinant)', 'C0751599'], ['beta 1 Interferon', 'C0005148'], ['Interferon beta-1b', 'C0244713'], ['beta 1a, Interferon', 'C0254119']]


In [12]:
query = "inf beta 1"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C0005148
Predicted label: beta 1 Interferon
Distance: 6.9799
Canonical form: beta 1 Interferon
Nearest 3:  [['beta 1 Interferon', 'C0005148'], ['beta 1a, Interferon', 'C0254119'], ['beta Interferon', 'C0015980'], ['Interferon beta (recombinant)', 'C0751599'], ['Interferon beta-1b', 'C0244713']]


In [13]:
query = "inf-beta"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C0015980
Predicted label: beta Interferon
Distance: 6.9821
Canonical form: beta Interferon
Nearest 3:  [['beta Interferon', 'C0015980'], ['Interferon beta (recombinant)', 'C0751599'], ['beta 1 Interferon', 'C0005148'], ['Interferon beta-1b', 'C0244713'], ['beta 1a, Interferon', 'C0254119']]


In [237]:
query = "sIL-13Rα2"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10, n_entities=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest entities: ", nearest_3_entities)

Predicted ID: -1
Predicted label: sIL-13Rα2
Distance: 12.3609
Canonical form: sIL-13Rα2
Nearest entities:  [['interleukin 13 antagonist E13K', 'C1097943'], ['interleukin-13 DM', 'C0967363'], ['ht-13-B', 'C4077538'], ['IL-13, human', 'C3658037'], ['CD2314', 'C4548772'], ['IL18R1 protein, human', 'C1447429'], ['Interleukin 13', 'C0214743'], ['KIT-13', 'C5853888'], ['IL36Ra protein, human', 'C4310349'], ['CDER167', 'C5773867']]
